# Validating the PageRank solver on a toy graph

## Purpose

Before running PageRank on the ~75,000-node AS-level Internet graph, we check
the implementation on graphs that are small enough to solve by hand. The real
graph has no known-good answer to compare against, so a wrong ranking there
would be guesswork to debug. A four-node graph whose exact PageRank vector is
known as rationals turns that guesswork into an exact comparison: if the solver
disagrees with the hand-derived vector on four nodes, the bug surfaces here
rather than hiding inside a large result.

This notebook uses two fixtures. The first has an out-link on every node, so
the transition matrix is stochastic and no special case applies. The second
contains a dangling node (no out-links), which is exactly what the project's
dangling-node treatment exists to handle, and which the first fixture
deliberately cannot exercise.

## The two fixtures

The fixtures, the exact ground-truth vector and the hand derivation live in
`tests/test_toy_graph.py`, where they are proved independently of any solver.
We import them from there so the notebook and the test suite validate against
the same numbers. The package itself is imported normally (it is installed
editable); only `tests/` needs to be added to the path, because it is not a
package.

`networkx` appears in this notebook for one purpose only: to *draw* the two
graphs. It never computes a number reported anywhere; every PageRank value
below comes from `src/pagerank/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# tests/ is not a package, so add it to the path to reuse the proven fixtures.
TESTS_DIR = Path.cwd().parent / "tests"
if str(TESTS_DIR) not in sys.path:
    sys.path.insert(0, str(TESTS_DIR))

from test_toy_graph import (
    ALPHA,
    DANGLING_EDGES,
    DANGLING_N,
    TOY_EDGES,
    TOY_N,
    TOY_PAGERANK,
    TOY_PAGERANK_EXACT,
)

# The package is imported normally.
from pagerank import from_edge_list, pagerank_power, ranking

alpha = float(ALPHA)
print(f"alpha = {alpha}")
print(f"toy edges:      {TOY_EDGES}")
print(f"dangling edges: {DANGLING_EDGES}")

In [ ]:
# Draw both fixtures. networkx is used here for layout and drawing only.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, edges, n, title in (
    (axes[0], TOY_EDGES, TOY_N, "TOY_EDGES (no dangling node)"),
    (axes[1], DANGLING_EDGES, DANGLING_N, "DANGLING_EDGES (node 3 dangling)"),
):
    g = nx.DiGraph()
    g.add_nodes_from(range(n))
    g.add_edges_from(edges)

    out_degree = dict(g.out_degree())
    colours = [
        "#d95f02" if out_degree[node] == 0 else "#7570b3" for node in g.nodes()
    ]
    pos = nx.circular_layout(g)
    nx.draw(
        g,
        pos,
        ax=ax,
        with_labels=True,
        node_color=colours,
        node_size=900,
        font_color="white",
        arrowsize=18,
    )
    ax.set_title(title)

fig.suptitle("Orange = dangling node (no out-links)")
plt.tight_layout()
plt.show()

## Building the graph

`from_edge_list` returns the sparse structure the solver needs: the
row-normalised transition matrix stored transposed (`P^T`), the out- and
in-degrees, and the dangling mask. Storing `P^T` is deliberate: the power step
multiplies `P^T x`, so this is the matrix the solver actually touches.

In the dense form below, column `i` holds node `i`'s out-distribution: its
non-zero entries are the `1 / out-degree(i)` weights and sum to 1. A dangling
node has an all-zero column, which is the leak the treatment has to repair.

In [ ]:
toy = from_edge_list(TOY_EDGES, TOY_N)

print("out-degree:", toy.out_degree.tolist())
print("in-degree: ", toy.in_degree.tolist())
print("dangling:  ", toy.dangling.tolist())
print("\nP^T (rows = head, cols = tail):")
print(np.round(toy.transpose.toarray(), 3))

In [ ]:
dang = from_edge_list(DANGLING_EDGES, DANGLING_N)

print("out-degree:", dang.out_degree.tolist())
print("in-degree: ", dang.in_degree.tolist())
print("dangling:  ", dang.dangling.tolist())
print("\nP^T (rows = head, cols = tail):")
print(np.round(dang.transpose.toarray(), 3))
print("\nColumn 3 is all zero: node 3 has no out-links, so its mass leaks.")

## Power iteration on the toy graph

Run the solver on `TOY_EDGES` at `alpha = 0.85` with a tight tolerance, and
compare against the exact rational vector from the fixture. The absolute error
column shows the agreement to roughly machine precision. Node 3 has no incoming
edge, so it can only hold the teleport mass `(1 - alpha) / n = 0.0375`, which is
a value we can check by eye.

In [ ]:
result = pagerank_power(toy, alpha=alpha, tol=1e-12)
exact = np.array([float(p) for p in TOY_PAGERANK_EXACT])

table = pd.DataFrame(
    {
        "node": range(TOY_N),
        "computed": result.vector,
        "exact": exact,
        "abs_error": np.abs(result.vector - exact),
    }
)
print(table.to_string(index=False))
print(f"\nconverged: {result.converged} in {result.iterations} iterations")
print(f"sum of vector: {result.vector.sum():.15f}")
print(f"ranking (desc): {ranking(result.vector).tolist()}")
print(
    f"node 3 computed = {result.vector[3]:.6f}, "
    f"(1 - alpha)/n = {(1 - alpha) / TOY_N:.6f}"
)

## Convergence

The damped update is a contraction with factor `alpha`: each step shrinks the
distance to the fixed point by at most a factor of `alpha`, so the L1 residual
decays at least geometrically at rate `alpha`. On a semilog axis that is a
straight line of slope `log(alpha)`, and the reference line `alpha**k` (scaled
to the first residual) sits at or above the measured curve.

In [ ]:
residuals = np.array(result.residuals)
steps = np.arange(1, len(residuals) + 1)
reference = residuals[0] * alpha ** (steps - 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(steps, residuals, "o-", label="measured L1 residual")
ax.semilogy(steps, reference, "--", label=r"$\alpha^k$ (scaled to first residual)")
ax.set_xlabel("iteration $k$")
ax.set_ylabel(r"$\|x_{k+1} - x_k\|_1$")
ax.set_title("Residual decays at rate bounded by alpha")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Dangling nodes

Node 3 of the dangling fixture has no out-links. Without treatment, the plain
update `x_new = alpha * P^T x + (1 - alpha) * v` throws away the mass that lands
on node 3 every iteration, because its column of `P^T` is zero. The total mass
then falls below 1 and keeps falling. The six-line loop below is written inline
to show the leak; it is deliberately *not* part of the package, which only ships
the corrected solver.

In [ ]:
# Plain update WITHOUT dangling treatment, written inline to show the leak.
v = np.full(DANGLING_N, 1.0 / DANGLING_N)
x = v.copy()
leaked_mass = [x.sum()]
for _ in range(20):
    x = alpha * (dang.transpose @ x) + (1 - alpha) * v
    leaked_mass.append(x.sum())

for k, mass in enumerate(leaked_mass[:8]):
    print(f"iteration {k:2d}: sum(x) = {mass:.6f}")
print("...")
print(f"iteration {len(leaked_mass) - 1:2d}: sum(x) = {leaked_mass[-1]:.6f}")
print("\nMass leaks away because node 3's mass has nowhere to go.")

The project solver redistributes the dangling mass `alpha * m` along the
teleport vector *inside* each update, which is algebraically identical to
replacing node 3's row with the teleport distribution. That keeps the matrix
stochastic, so the vector sums to 1 at every step. The bar chart contrasts the
leaked vector (not a distribution) with the solver's vector (a distribution).

In [ ]:
treated = pagerank_power(dang, alpha=alpha, tol=1e-12)

fig, ax = plt.subplots(figsize=(7, 4))
nodes = np.arange(DANGLING_N)
width = 0.38
ax.bar(nodes - width / 2, x, width, label=f"no treatment (sum={x.sum():.3f})")
ax.bar(
    nodes + width / 2,
    treated.vector,
    width,
    label=f"project solver (sum={treated.vector.sum():.3f})",
)
ax.set_xticks(nodes)
ax.set_xlabel("node")
ax.set_ylabel("PageRank")
ax.set_title("Dangling-node treatment conserves mass")
ax.legend()
plt.tight_layout()
plt.show()

print(f"solver ranking (desc): {ranking(treated.vector).tolist()}")
print("node 0 is unreachable, so it ranks last.")

In [ ]:
# Equivalence check: the solver's vector equals the leaked vector normalised.
# This is the fixture's convention-equivalence proof seen numerically.
normalised_leak = x / x.sum()
max_diff = np.abs(treated.vector - normalised_leak).max()
print(f"max |solver - normalised_leak| = {max_diff:.2e}")
assert max_diff < 1e-12
print("The solver and the normalised leaked vector agree to 1e-12.")

## PageRank in the course's notation

The solver above iterates PageRank in the project's column form, storing
`P^T`. The course lesson builds the same quantity as a chain of dense
matrices in *row* form, on graphs small enough to write out: the
adjacency matrix `A`, the row-stochastic hyperlink matrix
`H = diag(D)^-1 A`, the stochastic matrix `S` that repairs dangling rows,
the Google matrix `G = alpha S + (1 - alpha)/N 1 1^T`, and PageRank as the
dominant left eigenvector of `G`. The dense helpers in `pagerank.dense`
implement that chain exactly; they are for small graphs only and are never
applied to the real AS graph.

Below, the chain is built on the lecture's five-node example (nodes
V, W, X, Y, Z; V is dangling) at `alpha = 0.85`, and the eigenvector of
`G` is placed next to the project's sparse solver on the same graph.

In [ ]:
from pagerank.dense import (
    adjacency_matrix,
    hyperlink_matrix,
    stochastic_matrix,
    google_matrix,
    pagerank_eigenvector,
)

# The lecture's 5-node 'general case': V W X Y Z, V dangling, alpha 0.85.
FIVE_EDGES = [(1, 2), (1, 3), (2, 3), (2, 4), (3, 4),
              (4, 0), (4, 1), (4, 2), (4, 3)]
FIVE_N = 5
FIVE_LABELS = ['V', 'W', 'X', 'Y', 'Z']
a_mat = adjacency_matrix(FIVE_EDGES, FIVE_N)
h_mat = hyperlink_matrix(a_mat)
s_mat = stochastic_matrix(h_mat)
g_mat = google_matrix(s_mat, alpha=0.85)

np.set_printoptions(precision=4, suppress=True)
print('A =\n', a_mat)
print('\nH = diag(D)^-1 A =\n', h_mat)
print('\nS (dangling row of V repaired to 1/N) =\n', s_mat)
print('\nG = alpha S + (1 - alpha)/N 1 1^T =\n', g_mat)

pi_dense = pagerank_eigenvector(g_mat)
five_graph = from_edge_list(FIVE_EDGES, FIVE_N)
pi_sparse = pagerank_power(five_graph, alpha=0.85, tol=1e-13).vector
side_by_side = pd.DataFrame({
    'node': FIVE_LABELS,
    'eigenvector(G)': pi_dense,
    'power method': pi_sparse,
    'abs_diff': np.abs(pi_dense - pi_sparse),
})
print('\n', side_by_side.to_string(index=False))

# The project stores H transposed: graph.transpose == H^T. Check it here.
assert np.allclose(five_graph.transpose.toarray(), h_mat.T, atol=1e-12)
print('\ngraph.transpose == H^T:', True)

Why each step of `A -> H -> S -> G` exists:

- **`A -> H`** divides each row of `A` by the node's out-degree, so a
  surfer leaves a node uniformly along its out-links. A dangling node (V
  here) has an all-zero row, which leaves `H` only sub-stochastic.
- **`H -> S`** replaces each dangling row with the uniform distribution
  `1/N`, so every row of `S` sums to 1.
- **`S -> G`** mixes in uniform teleportation. With `0 < alpha < 1` every
  entry of `G` is strictly positive, so `G` is *primitive* (irreducible
  and aperiodic); by Perron-Frobenius it then has a unique positive
  dominant left eigenvector with eigenvalue 1, which is PageRank.

The mapping to the code. `H` is the row-stochastic hyperlink matrix and
the project stores its transpose, so `graph.transpose == H^T` (asserted
above on this example). The course's matrix-free update

    pi^(k+1)T = alpha pi^(k)T H + (alpha pi^(k)T w + 1 - alpha) 1^T / N

is, transposed, exactly the update in `pagerank_power`:
`x_new = alpha P^T x + (alpha m + 1 - alpha) v`, with `m = w^T x` the mass
on dangling nodes and `v = 1/N` the uniform teleport vector.

Why the project never builds `G` on the real graph: `G` is dense, so for
the AS graph it would have `80_510**2` entries (billions), while the
stored `H` has only 884,790 non-zeros. The matrix-free update multiplies
the sparse `P^T` and adds the rank-one teleport term, never forming `G`.

## The lecture's examples, reproduced

The project's solver run on all four graphs solved in the lesson, against
the values printed there. The printed values are rounded to the precision
shown on the slides (four decimals for the five-node example, three for the
two exercises), so small absolute differences are rounding, not
disagreement.

In [ ]:
# Edge lists and node orders as given; printed values as shown in the lesson.
EXAMPLES = {
    '5-node': dict(
        edges=FIVE_EDGES, n=5, alpha=0.85, labels=FIVE_LABELS,
        printed=[0.1219, 0.1219, 0.1737, 0.2475, 0.3350]),
    '6-node': dict(
        edges=[(0, 1), (0, 2), (1, 0), (1, 2), (1, 3), (3, 4), (3, 5),
               (4, 3), (4, 5), (5, 3)],
        n=6, alpha=0.9, labels=list('ABCDEF'),
        printed=[0.037, 0.042, 0.054, 0.383, 0.197, 0.286]),
    '8-node': dict(
        edges=[(0, 1), (0, 2), (1, 0), (1, 5), (2, 3), (2, 4), (3, 2),
               (3, 4), (4, 2), (5, 6), (5, 7), (6, 5), (6, 7), (7, 5)],
        n=8, alpha=0.9, labels=list('ABCDEFGH'),
        printed=[0.023, 0.023, 0.212, 0.108, 0.157, 0.212, 0.108, 0.157]),
}
rows = []
for name, ex in EXAMPLES.items():
    solver = pagerank_power(
        from_edge_list(ex['edges'], ex['n']), alpha=ex['alpha'], tol=1e-13
    ).vector
    for label, printed, got in zip(ex['labels'], ex['printed'], solver):
        rows.append({
            'example': name, 'alpha': ex['alpha'], 'node': label,
            'printed': printed, 'solver': got,
            'abs_diff': abs(printed - got),
        })
examples_tbl = pd.DataFrame(rows)
print(examples_tbl.to_string(index=False,
      formatters={'solver': '{:.4f}'.format, 'abs_diff': '{:.4f}'.format}))
print('\nlargest absolute difference from any printed value:',
      f"{examples_tbl['abs_diff'].max():.4f}")

The lecture's four-node *simple case* (no teleportation) has an exact
rational solution, and the lesson uses it to show that in-degree and
PageRank can rank a graph differently.

In [ ]:
from fractions import Fraction

# W X Y Z, every node has out-links, so with no teleportation PageRank is
# the stationary distribution pi H = pi. Exact solution from the lesson.
FOUR_EDGES = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 0), (3, 1), (3, 2)]
FOUR_LABELS = ['W', 'X', 'Y', 'Z']
four_exact = (Fraction(4, 31), Fraction(6, 31), Fraction(9, 31), Fraction(12, 31))
a_four = adjacency_matrix(FOUR_EDGES, 4)
in_degree = a_four.sum(axis=0).astype(int)
four_tbl = pd.DataFrame({
    'node': FOUR_LABELS,
    'pagerank (exact)': [str(f) for f in four_exact],
    'pagerank (float)': [float(f) for f in four_exact],
    'in_degree': in_degree,
})
print(four_tbl.to_string(index=False))
pr_order = [FOUR_LABELS[i] for i in np.argsort([-float(f) for f in four_exact])]
deg_order = [FOUR_LABELS[i] for i in np.argsort(-in_degree)]
print('\nranked by PageRank: ', pr_order)
print('ranked by in-degree:', deg_order)
print('the two orderings differ:', pr_order != deg_order)

By in-degree Y leads (three in-links); by PageRank Z leads (`12/31`), even
though Z has two in-links. PageRank counts *where* the links come from, not
only how many: Z is pointed at by nodes that are themselves well-ranked.
This is the same mechanism the in-degree comparison on the real AS graph
turns on.

## What this notebook established

- The solver reproduces the hand-derived PageRank of `TOY_EDGES` to roughly
  machine precision, and places the unreachable node 3 at exactly `(1-alpha)/n`.
- The L1 residual decays geometrically at a rate bounded by `alpha`, matching
  the contraction-factor argument.
- Without dangling-node treatment, mass leaks every iteration and the vector
  stops being a probability distribution.
- The project's treatment conserves mass, and its output equals the leaked
  vector normalised at the end to 1e-12, confirming numerically that the two
  dangling conventions coincide.